# Multimodal RAG with DeepSeek
This notebook uses the structure of `test_rag_on_local_llm.ipynb` (LangChain) but adapts it for 3 types of data: Text, Images, and Sheets.

### Answering your question about DeepSeek OCR:
*"I want deepseek ocr to split the text and image and sheet(is it possible)"*
**Answer:** DeepSeek OCR is a Vision-Language Model. It receives an **image** as input and outputs structured text (like markdown). 
It cannot natively take a `.pdf` or `.docx` and split it into separate text, image, and Excel files. To split a document into its components, you typically use a tool like **Docling** or **PyMuPDF**. 
However, what we *can* do (and have done below) is:
1. Load **Text** natively.
2. Load **Sheets/CSV** via pandas natively (converts to markdown tables).
3. Use **DeepSeek OCR** to extract text and data from **Images**. 

Once everything is converted to text/markdown, it is ingested into ChromaDB using standard `OllamaEmbeddings`, and we use `deepseekv3:latest` for the final RAG answer.


In [ ]:
!pip install PyMuPDF langchain-text-splitters langchain-ollama langchain-community langchain-core chromadb sentence-transformers pillow pandas openpyxl

In [ ]:
import os
import base64
import requests
import shutil
import pandas as pd
from glob import glob
from PIL import Image

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
import chromadb

In [ ]:
# ==========================================
# 階段一：模型與伺服器設定 (Configuration)
# ==========================================

OLLAMA_SERVER_URL = "http://localhost:11434"

# Use your requested models
OLLAMA_VISION_MODEL = "deepseek-ocr"
OLLAMA_CHAT_MODEL = "deepseekv3:latest"
EMBEDDING_MODEL = "embeddinggemma" # Using embeddinggemma as in your test notebook

print(f"Vision Model: {OLLAMA_VISION_MODEL}")
print(f"Chat Model: {OLLAMA_CHAT_MODEL}")
print(f"Embedding Model: {EMBEDDING_MODEL}")

In [ ]:
# ==========================================
# 階段二：3 Types of Data Loaders (Text, Sheet, Image via OCR)
# ==========================================

def load_text(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return f.read()

def load_sheet(file_path):
    """Converts CSV or Excel into a markdown table."""
    if file_path.endswith('.csv'):
        df = pd.read_csv(file_path)
    else:
        df = pd.read_excel(file_path)
    return df.to_markdown(index=False)

def deepseek_ocr_image(image_path):
    """Uses DeepSeek OCR to extract text/tables from an image."""
    endpoint = f"{OLLAMA_SERVER_URL}/api/generate"
    
    with open(image_path, "rb") as f:
        img_b64 = base64.b64encode(f.read()).decode("utf-8")
        
    payload = {
        "model": OLLAMA_VISION_MODEL,
        "prompt": "Extract all text and tabular data from this image perfectly. If there is a table, format it as a markdown table.",
        "images": [img_b64],
        "stream": False
    }
    
    try:
        resp = requests.post(endpoint, json=payload)
        resp.raise_for_status()
        return resp.json()["response"].strip()
    except Exception as e:
        print(f"Error calling Ollama DeepSeek OCR on {image_path}: {e}")
        return "" 

In [ ]:
# ==========================================
# 階段三：資料處理與向量庫建立 (Data Ingestion)
# ==========================================

def load_all_documents(folder_path):
    docs = []
    
    # 1. Text Data
    for ext in ['*.txt', '*.md']:
        for file_path in glob(os.path.join(folder_path, ext)):
            print(f"Loading Text: {file_path}")
            content = load_text(file_path)
            docs.append(Document(page_content=content, metadata={"source": file_path, "type": "text"}))
            
    # 2. Sheet Data
    for ext in ['*.csv', '*.xlsx']:
        for file_path in glob(os.path.join(folder_path, ext)):
            print(f"Loading Sheet: {file_path}")
            content = load_sheet(file_path)
            docs.append(Document(page_content=content, metadata={"source": file_path, "type": "sheet"}))
            
    # 3. Image Data
    for ext in ['*.png', '*.jpg', '*.jpeg']:
        for file_path in glob(os.path.join(folder_path, ext)):
            print(f"Loading Image via DeepSeek OCR: {file_path}")
            content = deepseek_ocr_image(file_path)
            if content:
                docs.append(Document(page_content=content, metadata={"source": file_path, "type": "image"}))
            
    return docs

folder = "multimodal_data"
os.makedirs(folder, exist_ok=True)
print(f"Created folder '{folder}'. Please place some text, images, and sheets there before running.")

In [ ]:
# 執行讀取並建立向量庫
docs = load_all_documents(folder)

if docs:
    # Chunking (切分文本)
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
    chunks = text_splitter.split_documents(docs)
    
    # Embedding Model
    embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)
    
    # 向量資料庫處理
    persist_dir = "./multimodal_vector_db"
    
    # Clear chromadb cache to prevent sqlite read-only errors in Jupyter
    chromadb.api.client.SharedSystemClient.clear_system_cache()
    
    if os.path.exists(persist_dir):
        shutil.rmtree(persist_dir)
        print("已刪除舊的向量庫，準備重建...")
        
    vector_db = Chroma.from_documents(
        chunks, 
        embeddings, 
        persist_directory=persist_dir
    )
    
    print(f"成功！已建立資料庫。總共切分了 {len(chunks)} 個區塊。")
else:
    print("目前資料夾沒有檔案。請放入檔案後重新執行。")
    # Backup for when running empty
    embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)
    persist_dir = "./multimodal_vector_db"
    vector_db = Chroma(persist_directory=persist_dir, embedding_function=embeddings)

In [ ]:
# ==========================================
# 階段四：檢索與大語言模型問答 (Retrieval & Generation)
# ==========================================

# 1. Setup Retriever (設定檢索器)
retriever = vector_db.as_retriever(search_kwargs={"k": 5})

# 2. Define the Local LLM (設定 DeepSeek v3)
llm = ChatOllama(
    model=OLLAMA_CHAT_MODEL,
    temperature=0  # 保持 0 讓回答精準
)

# 3. Create the Prompt
template = """
You are a helpful AI assistant. You have access to information extracted from multiple types of data: Text files, Spreadsheets (Tables), and Images.
Respond to the user's question accurately based ONLY on the provided context.

Context:
{context}

Question: {question}
Answer:
"""
prompt = ChatPromptTemplate.from_template(template)

# 4. Build the Chain using LCEL (組裝 RAG 流程)
qa_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# 5. 問答測試
question = "What information do we have regarding the battery brand?"
print(f"Question: {question}\n")
print(f"[{OLLAMA_CHAT_MODEL} 思考中...]\n")

response = qa_chain.invoke(question)
print(response)